In [28]:
import numpy as np 
import pandas as pd 
from scipy import stats
import json

In [30]:
import json

# Load parameters from file 1 (shelter population/length-of-stay model)
with open('/kaggle/input/datasets/micahluftig/monte-carlo-sim-data/simulation_params.json') as f:
    sim_params = json.load(f)

# Load parameters from file 2 (weather/pressure regression)
with open('/kaggle/input/datasets/micahluftig/weather-for-monte/weather_params.json') as f:
    weather_params = json.load(f)

# Unpack into individual variables
p_long_stay = sim_params['p_long_stay']
shape_short = sim_params['shape_short']
scale_short = sim_params['scale_short']
shape_long = sim_params['shape_long']
scale_long = sim_params['scale_long']
threshold = sim_params['threshold']
n_nb = sim_params['n_nb']
p_nb = sim_params['p_nb']
mean_intake = sim_params['mean_intake']

pct_change_per_mb = weather_params['pct_change_per_mb']
historical_avg_pressure = weather_params['historical_avg_pressure']

print("Simulation params loaded:", sim_params)
print("Weather params loaded:", weather_params)

Simulation params loaded: {'mean_intake': 45.42060561299852, 'p_long_stay': 0.03193521898552021, 'shape_short': 0.504179663713484, 'scale_short': 25.27816369975163, 'shape_long': 0.7357973293058672, 'scale_long': 126.78636081798182, 'threshold': 100, 'n_nb': 7.462422692196722, 'p_nb': 0.14111186388816566}
Weather params loaded: {'pct_change_per_mb': 0.01285, 'pressure_std': 5.99, 'coef_pressure_z': -0.0765, 'alpha_dispersion': 0.1621, 'historical_avg_pressure': 1016.5344165435747}


In [31]:
def adjusted_intake_mean(forecast_pressure, historical_avg_pressure, baseline_mean, pct_change_per_mb):
    pressure_diff = historical_avg_pressure - forecast_pressure  # positive = pressure drop = more intake
    multiplier = (1 + pct_change_per_mb) ** pressure_diff
    return baseline_mean * multiplier

def nb_params_from_mean(mean, dispersion_ratio):
    var = mean * dispersion_ratio
    p = mean / var
    n = mean * p / (1 - p)
    return n, p

# dispersion_ratio from file 1's variance/mean check (~7.09)
dispersion_ratio = var_intake / mean_intake if 'var_intake' in dir() else 7.09

That 100% result is actually really useful diagnostic information — it's telling you capacity=150 was just an arbitrary guess that's way below what this shelter realistically operates at, not that something's broken in your code.
Here's why: you can estimate the expected steady-state population using a queueing theory concept called Little's Law: population ≈ arrival_rate × average_length_of_stay. Let's check what your fitted distributions actually imply:

In [33]:
for pressure in [990, 1000, 1013, 1020, 1030]:
    adj_mean = adjusted_intake_mean(pressure, historical_avg_pressure, mean_intake, pct_change_per_mb)
    print(f"Pressure={pressure} mb → adjusted mean intake: {adj_mean:.2f} (baseline: {mean_intake:.2f})")

Pressure=990 mb → adjusted mean intake: 63.74 (baseline: 45.42)
Pressure=1000 mb → adjusted mean intake: 56.10 (baseline: 45.42)
Pressure=1013 mb → adjusted mean intake: 47.52 (baseline: 45.42)
Pressure=1020 mb → adjusted mean intake: 43.45 (baseline: 45.42)
Pressure=1030 mb → adjusted mean intake: 38.25 (baseline: 45.42)


This is the important sanity-check step — run this first and share the output. You want to see something sensible: lower pressure (storms) should push the adjusted mean above baseline, higher pressure should push it below, and the swings should be modest (a few percent either way for normal weather, not something wild like doubling or halving) given your ~1.3%/mb effect size.
Once that looks reasonable, we'll modify simulate_year() to accept a list of daily forecasted pressures and use adjusted_intake_mean() + nb_params_from_mean() to draw each day's arrivals instead of a fixed rate. Run that test loop and let me know what comes out



The direction is right (low pressure → higher intake, matching your original moon-phase finding), and the magnitude is reasonable — even a fairly extreme low-pressure day (990 mb, a meaningful storm system) pushes intake up about 40% above baseline, not some absurd multiple. That's believable behavior for a weather-driven adjustment.Now let's wire this into the actual simulation. The key change: instead of simulate_year() running for 365 days with a fixed arrival rate, we want a shorter function that takes a list of forecasted daily pressures and adjusts the arrival rate day-by-day:

In [35]:
def simulate_forecast_period(pressures, dispersion_ratio, capacity, starting_population=None):
    """
    pressures: list of forecasted daily pressure values (e.g., 7 days)
    starting_population: list of remaining LOS for animals already in shelter (optional)
    """
    current_animals = starting_population.copy() if starting_population else []
    daily_population = []
    days_over_capacity = 0

    for day_pressure in pressures:
        # Age existing animals
        current_animals = [los - 1 for los in current_animals if los - 1 > 0]

        # Adjust arrival rate based on this day's forecasted pressure
        adj_mean = adjusted_intake_mean(day_pressure, historical_avg_pressure, mean_intake, pct_change_per_mb)
        n_day, p_day = nb_params_from_mean(adj_mean, dispersion_ratio)
        new_arrivals = stats.nbinom.rvs(n_day, p_day)

        for _ in range(new_arrivals):
            if np.random.random() < p_long_stay:
                los = stats.gamma.rvs(shape_long, loc=threshold, scale=scale_long)
            else:
                los = stats.gamma.rvs(shape_short, loc=0, scale=scale_short)
            current_animals.append(los)

        population_today = len(current_animals)
        daily_population.append(population_today)
        if population_today > capacity:
            days_over_capacity += 1

    return days_over_capacity, daily_population, current_animals

Key design choice worth flagging: this function accepts starting_population — a list of "remaining days" for animals already in the shelter before the forecast window starts. This matters a lot for a realistic short-term tool: a 7-day forecast shouldn't assume the shelter starts empty, it should start from something close to your real steady-state population (~841, from Little's Law) or ideally an actual current count if you ever hook this up to real data.
Test it with a plausible starting population and a sample forecast:

In [36]:
# Rough starting point: use your long-run simulation's typical ending population
# (or just approximate with steady-state distribution for now)
np.random.seed(42)
sample_forecast = [1013, 1005, 995, 990, 998, 1008, 1015]  # a week, dipping into a storm

results = [simulate_forecast_period(sample_forecast, dispersion_ratio, capacity=1000) 
           for _ in range(1000)]
days_over = [r[0] for r in results]

print(f"% of simulated weeks with overflow: {(np.array(days_over) > 0).mean()*100:.1f}%")

% of simulated weeks with overflow: 0.0%


That 0% is exactly the artifact I flagged — the simulation is starting from an empty shelter (0 animals) and only running 7 days, so even with elevated intake from the storm, there's no way it fills up to anywhere near capacity=1000 in just a week starting from zero. This isn't a real result yet — it's just confirming the function runs, but the starting conditions aren't realistic.
Let's fix this by generating a realistic starting population. The cleanest way: run one long simulation to steady-state, then just grab the population at some point once it's stabilized, and use that as your realistic "current shelter state."


This runs a full year at baseline (no weather adjustment) just to "warm up" the shelter to a realistic ongoing population, then hands you that list of remaining-LOS values as day-1 conditions for your 7-day forecast simulation.
Now re-run the forecast test using that as the starting point:


One subtlety: since starting_population is the same list every time right now (not re-randomized per simulation), all 1,000 runs start from an identical snapshot — which is actually fine and arguably more realistic for a real app (you'd start from today's actual known population, not a random one), but worth knowing that only the forecast period itself is what's varying across simulations, not the starting conditions.
Run that and share the overflow percentage — that should give you a much more meaningful number now that the shelter starts "full" rather than empty.

In [37]:
def generate_steady_state_population(n_days=365, dispersion_ratio=7.09):
    current_animals = []
    for day in range(n_days):
        current_animals = [los - 1 for los in current_animals if los - 1 > 0]
        new_arrivals = stats.nbinom.rvs(n_nb, p_nb)  # baseline rate, no weather adjustment
        for _ in range(new_arrivals):
            if np.random.random() < p_long_stay:
                los = stats.gamma.rvs(shape_long, loc=threshold, scale=scale_long)
            else:
                los = stats.gamma.rvs(shape_short, loc=0, scale=scale_short)
            current_animals.append(los)
    return current_animals

np.random.seed(42)
starting_population = generate_steady_state_population()
print(f"Starting population size: {len(starting_population)}")

Starting population size: 880


In [38]:
results = [simulate_forecast_period(sample_forecast, dispersion_ratio, capacity=1000, 
                                      starting_population=starting_population) 
           for _ in range(1000)]
days_over = [r[0] for r in results]

print(f"% of simulated weeks with overflow: {(np.array(days_over) > 0).mean()*100:.1f}%")

% of simulated weeks with overflow: 0.2%


That's a much more meaningful number, and it actually makes sense: your steady-state population is ~841, capacity here is 1000 (about 160 above average, similar to where your earlier sweep showed risk dropping toward zero), so a 0.2% overflow rate over a single storm week is a believable, if modest, result.
A few good next moves to make this analysis more useful and interesting:
1. Compare against a no-storm baseline — to actually show the weather effect is doing something

In [39]:
flat_forecast = [1013] * 7  # average pressure, no storm
results_flat = [simulate_forecast_period(flat_forecast, dispersion_ratio, capacity=1000,
                                           starting_population=starting_population)
                for _ in range(1000)]
days_over_flat = [r[0] for r in results_flat]
print(f"% of weeks with overflow (no storm): {(np.array(days_over_flat) > 0).mean()*100:.1f}%")

% of weeks with overflow (no storm): 0.2%


Good catch on running this — but notice the storm and no-storm scenarios came back identical (0.2% both times). That's a real signal, not a coincidence: at capacity=1000, you have so much buffer above the ~841 steady-state population that even a storm-driven bump in intake isn't enough to meaningfully change the odds of overflow. The weather effect exists, but it's getting swamped by how much slack is in the system at this capacity.
This is exactly why I suggested testing at a lower capacity — that's where the storm's effect should actually become visible. Let's try capacity=900:

In [40]:
results_storm_900 = [simulate_forecast_period(sample_forecast, dispersion_ratio, capacity=900,
                                                starting_population=starting_population)
                      for _ in range(1000)]
days_over_storm_900 = [r[0] for r in results_storm_900]
print(f"% of weeks with overflow (storm, capacity=900): {(np.array(days_over_storm_900) > 0).mean()*100:.1f}%")

results_flat_900 = [simulate_forecast_period(flat_forecast, dispersion_ratio, capacity=900,
                                               starting_population=starting_population)
                     for _ in range(1000)]
days_over_flat_900 = [r[0] for r in results_flat_900]
print(f"% of weeks with overflow (no storm, capacity=900): {(np.array(days_over_flat_900) > 0).mean()*100:.1f}%")

% of weeks with overflow (storm, capacity=900): 61.2%
% of weeks with overflow (no storm, capacity=900): 32.5%


That's a big jump from the capacity=1000 case (0.2%) — makes sense since 900 is much closer to your ~841 steady-state average, so normal variation alone puts you at real risk of overflow, and the storm is now landing in a range where it actually matters.
But we need the no-storm number at capacity=900 too, to know how much of that 61.2% is the storm versus just being at a risky capacity in general:

In [41]:
results_flat_900 = [simulate_forecast_period(flat_forecast, dispersion_ratio, capacity=900,
                                               starting_population=starting_population)
                     for _ in range(1000)]
days_over_flat_900 = [r[0] for r in results_flat_900]
print(f"% of weeks with overflow (no storm, capacity=900): {(np.array(days_over_flat_900) > 0).mean()*100:.1f}%")

% of weeks with overflow (no storm, capacity=900): 34.3%
